# Código de proceso ETL

In [39]:
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from sqlalchemy import text
import os
from dotenv import load_dotenv
from sqlalchemy.pool import NullPool

# 1. Conexión Base de Datos

In [40]:
# Load environment variables from .env
load_dotenv(dotenv_path=".env", override=True)

USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

print("USER:", USER)
print("HOST:", HOST)
print("PORT:", PORT)

DATABASE_URL = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}?sslmode=require"

engine = create_engine(
    DATABASE_URL,
    poolclass=NullPool
)

try:
    with engine.connect() as connection:
        print("Connection successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

USER: None
HOST: None
PORT: None


ValueError: invalid literal for int() with base 10: 'None'

# 2. Extracción de datos

In [ ]:
csv_file = "customer_shopping_data.csv"
df = pd.read_csv(csv_file)
df.head()


,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,5/8/2022,Kanyon
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,12/12/2021,Forum Istanbul
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,9/11/2021,Metrocity
3,I173702,C988172,Female,66,Shoes,5,3000.85,Credit Card,16/05/2021,Metropol AVM
4,I337046,C189076,Female,53,Books,4,60.60,Cash,24/10/2021,Kanyon


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      99457 non-null  object 
 1   customer_id     99457 non-null  object 
 2   gender          99457 non-null  object 
 3   age             99457 non-null  int64  
 4   category        99457 non-null  object 
 5   quantity        99457 non-null  int64  
 6   price           99457 non-null  float64
 7   payment_method  99457 non-null  object 
 8   invoice_date    99457 non-null  object 
 9   shopping_mall   99457 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 7.6+ MB


# 3. Trasformación

In [ ]:
"""
Covertir la letra inicial de los identificadores en un número espécifico 
para mantener esa diferenciación 
y convertirlos en tipo enteros
"""
df['invoice_no'] = df['invoice_no'].replace("^I", "1", regex=True).astype(int)
df['customer_id'] = df['customer_id'].replace("^C", "1", regex=True).astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      99457 non-null  int64  
 1   customer_id     99457 non-null  int64  
 2   gender          99457 non-null  object 
 3   age             99457 non-null  int64  
 4   category        99457 non-null  object 
 5   quantity        99457 non-null  int64  
 6   price           99457 non-null  float64
 7   payment_method  99457 non-null  object 
 8   invoice_date    99457 non-null  object 
 9   shopping_mall   99457 non-null  object 
dtypes: float64(1), int64(4), object(5)
memory usage: 7.6+ MB


In [ ]:
#Se convierte la fecha de compra en un tipo fecha
df['invoice_date'] = pd.to_datetime(df['invoice_date'], format="%d/%m/%Y")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99457 entries, 0 to 99456
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   invoice_no      99457 non-null  int64         
 1   customer_id     99457 non-null  int64         
 2   gender          99457 non-null  object        
 3   age             99457 non-null  int64         
 4   category        99457 non-null  object        
 5   quantity        99457 non-null  int64         
 6   price           99457 non-null  float64       
 7   payment_method  99457 non-null  object        
 8   invoice_date    99457 non-null  datetime64[ns]
 9   shopping_mall   99457 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(4), object(4)
memory usage: 7.6+ MB


In [ ]:
#Se cambia el nombre de las columnas de ID's para mejor claridad
df.rename(columns={'customer_id': 'id_customer'}, inplace=True)
df.rename(columns={'invoice_no': 'id_invoice'}, inplace=True)
df.head()

,id_invoice,id_customer,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall
0,1138884,1241288,Female,28,Clothing,5,1500.40,Credit Card,2022-08-05,Kanyon
1,1317333,1111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul
2,1127801,1266599,Male,20,Clothing,1,300.08,Cash,2021-11-09,Metrocity
3,1173702,1988172,Female,66,Shoes,5,3000.85,Credit Card,2021-05-16,Metropol AVM
4,1337046,1189076,Female,53,Books,4,60.60,Cash,2021-10-24,Kanyon


# Creación del modelo estrella

### Creamos la tabla dim_customer

In [ ]:
dim_customer = df[["id_customer", "gender", "age"]].copy()
dim_customer.head()

,id_customer,gender,age
0,1241288,Female,28
1,1111565,Male,21
2,1266599,Male,20
3,1988172,Female,66
4,1189076,Female,53


### Ahora creamos la tabla dim_category

In [41]:
df.category.unique()

array(['Clothing', 'Shoes', 'Books', 'Cosmetics', 'Food & Beverage',
       'Toys', 'Technology', 'Souvenir'], dtype=object)

In [42]:
dim_category = df[["category"]].drop_duplicates().reset_index(drop=True)
dim_category["id_category"] = dim_category.index + 1
dim_category.head()

,category,id_category
0,Clothing,1
1,Shoes,2
2,Books,3
3,Cosmetics,4
4,Food & Beverage,5


### Creamos la tabla dim_date

In [68]:
df.invoice_date.unique()

<DatetimeArray>
['2022-08-05 00:00:00', '2021-12-12 00:00:00', '2021-11-09 00:00:00',
 '2021-05-16 00:00:00', '2021-10-24 00:00:00', '2022-05-24 00:00:00',
 '2022-03-13 00:00:00', '2021-01-13 00:00:00', '2021-11-04 00:00:00',
 '2021-08-22 00:00:00',
 ...
 '2022-06-12 00:00:00', '2021-02-01 00:00:00', '2021-07-02 00:00:00',
 '2021-03-07 00:00:00', '2021-05-18 00:00:00', '2021-01-29 00:00:00',
 '2021-01-01 00:00:00', '2021-08-12 00:00:00', '2022-01-08 00:00:00',
 '2021-05-11 00:00:00']
Length: 797, dtype: datetime64[ns]

In [70]:
dim_date = df[["invoice_date"]].drop_duplicates().reset_index(drop=True)
dim_date["id_date"] = dim_date.index + 1
dim_date["day"] = dim_date["invoice_date"].dt.day
dim_date["month"] = dim_date["invoice_date"].dt.month
dim_date["year"] = dim_date["invoice_date"].dt.year
dim_date.head()

,invoice_date,id_date,day,month,year
0,2022-08-05,1,5,8,2022
1,2021-12-12,2,12,12,2021
2,2021-11-09,3,9,11,2021
3,2021-05-16,4,16,5,2021
4,2021-10-24,5,24,10,2021


### Creamos la tabla dim_store

In [44]:
df.shopping_mall.unique()

array(['Kanyon', 'Forum Istanbul', 'Metrocity', 'Metropol AVM',
       'Istinye Park', 'Mall of Istanbul', 'Emaar Square Mall',
       'Cevahir AVM', 'Viaport Outlet', 'Zorlu Center'], dtype=object)

In [50]:
dim_store = df[["shopping_mall"]].drop_duplicates().reset_index(drop=True)
dim_store["id_mall"] = dim_store.index + 1
dim_store.head(10)

,shopping_mall,id_mall
0,Kanyon,1
1,Forum Istanbul,2
2,Metrocity,3
3,Metropol AVM,4
4,Istinye Park,5
5,Mall of Istanbul,6
6,Emaar Square Mall,7
7,Cevahir AVM,8
8,Viaport Outlet,9
9,Zorlu Center,10


### Creamos la tabla dim_payment

In [51]:
df.payment_method.unique()

array(['Credit Card', 'Debit Card', 'Cash'], dtype=object)

In [52]:
dim_payment = df[["payment_method"]].drop_duplicates().reset_index(drop=True)
dim_payment["id_payment_method"] = dim_payment.index + 1
dim_payment.head()

,payment_method,id_payment_method
0,Credit Card,1
1,Debit Card,2
2,Cash,3


### Creamos la tabla de hechos

In [78]:
facts_sales = df.copy()
facts_sales = facts_sales.merge(dim_category, on='category', how='left')
facts_sales = facts_sales.merge(dim_date[['id_date', 'invoice_date']], on='invoice_date', how='left') 
facts_sales = facts_sales.merge(dim_store, on='shopping_mall', how='left') 
facts_sales = facts_sales.merge(dim_payment, on='payment_method', how='left') 
facts_sales = facts_sales.drop(columns=['category', 'payment_method', 'shopping_mall', 'invoice_date', 'gender', 'age'])
facts_sales.head()

,id_invoice,id_customer,quantity,price,id_category,id_date,id_mall,id_payment_method
0,1138884,1241288,5,1500.40,1,1,1,1
1,1317333,1111565,3,1800.51,2,2,2,2
2,1127801,1266599,1,300.08,1,3,3,3
3,1173702,1988172,5,3000.85,2,4,4,1
4,1337046,1189076,4,60.60,3,5,1,3


## Enviamos las tablas a la base de datos

In [ ]:
dim_customer.to_sql("dim_customer", engine, if_exists="replace", index=False)

NameError: name 'engine' is not defined

In [ ]:
dim_category.to_sql("dim_category", engine, if_exists="replace", index=False)

In [ ]:
dim_date.to_sql("dim_date", engine, if_exists="replace", index=False)

In [ ]:
dim_store.to_sql("dim_store", engine, if_exists="replace", index=False)

In [ ]:
dim_payment.to_sql("dim_payment", engine, if_exists="replace", index=False)

In [ ]:
facts_sales.to_sql("facts_sales", engine, if_exists="replace", index=False)